# 01 · Active loop — dry run (no hardware)

Runs the **exact same active-learning loop** used on the microscope, but against a
simulated `VirtualInstrument`. Use this to understand the workflow and sanity-check
parameters before spending microscope time — nothing here touches Igor.

The loop: pick a detection position → 'measure' a complex tune there → update the
low-rank reconstruction → pick the next position by D-optimal design → stop when the
D-NS confidence interval is small enough.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from activemodemap import VirtualInstrument, LowRankModeMap, plot_state

## Parameters

In [ ]:
# common tune window (Hz) and accessible laser-position grid (um)
freq_grid = np.linspace(254e3, 449e3, 400)
L_um      = 225.0                      # cantilever length (for reporting)
x_grid    = np.arange(L_um - 40, L_um + 0.01, 1.0)   # last ~40 um near the tip

RANK          = 5      # spatial basis dimension (map is low rank)
MAX_POSITIONS = 10
DNS_CI_TOL_UM = 1.0    # stop when D-NS 95% CI drops below this
SEEDS_UM      = [x_grid[0], x_grid[len(x_grid)//2], x_grid[-1]]

In [ ]:
inst = VirtualInstrument(freq_grid_Hz=freq_grid, noise_floor=0.08,
                         spot_fwhm_um=4.0, rng=np.random.default_rng(1))
print(f'(hidden) true D-NS = {inst.dns_um_from_end:.2f} um from the free end')

## Run the loop (live plot each step)

In [ ]:
mm = LowRankModeMap(x_grid, freq_grid, rank=RANK, seeds_um=SEEDS_UM,
                    dns_ci_tol_um=DNS_CI_TOL_UM)
rec = None
for step in range(MAX_POSITIONS):
    x = mm.next_position()
    f, Z, meta = inst.measure_at(x)          # <-- hardware call is swapped in here
    mm.add_measurement(x, f, Z)
    if mm.n >= mm.min_positions:
        rec = mm.reconstruct(nboot=150)
        dns_end = L_um - rec['dns']
        print(f'N={mm.n:2d}  x={x:6.1f} um  ->  D-NS = {dns_end:5.2f} um from end'
              f'  (95% CI {rec["dns_ci"]:.2f} um)   converged={mm.converged()}')
        if mm.converged():
            print('\nConverged.'); break

In [ ]:
ax = plot_state(mm, rec)
plt.tight_layout(); plt.show()
print(f'positions used: {mm.n}   true D-NS {inst.dns_um_from_end:.2f} um   '
      f'recovered {L_um-rec["dns"]:.2f} +/- {rec["dns_ci"]/2:.2f} um')